# Step 2b: Language Detection & Filtering
## YouTube Fast Fashion Intelligence Engine — CSCI370

**Why this step?**

When we inspected the dataset, we noticed comments in many languages — Italian, Spanish, Portuguese, French, German, Russian, and more.
About **23% of our comments (~9,000) are non-English**.

**Our decision: Remove non-English comments.**

Reasons:
- All our NLP tools (VADER, spaCy, BERTopic, KeyBERT) are trained on English text
- Translating 9,000 comments introduces noise and inaccuracy
- We still have ~30,000 English comments — a strong dataset
- This is a valid, standard research scoping decision: *"Analysis is limited to English-language comments"*

**Tool we use: `langdetect`**
A Python library that identifies the language of any text. It returns a language code like `en` (English), `es` (Spanish), `it` (Italian), etc.


In [1]:
!pip install langdetect -q


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load Libraries and Data

In [2]:
import pandas as pd
from langdetect import detect, LangDetectException, DetectorFactory

# IMPORTANT: Set a random seed so language detection gives consistent results
# langdetect uses randomness internally — without this, the same text could
# get different labels on different runs
DetectorFactory.seed = 42

# Load the dataset from Step 2 (with sentiment columns)
df = pd.read_csv('youtube_comments_sentiment.csv')

print(f"Loaded {len(df):,} comments")
df.head(3)

Loaded 39,422 comments


,author,updated_at,like_count,text,video_id,public,text_clean,sentiment_score,sentiment_label,sentiment_source
0,@joaquinrodriguezvillegas366,2026-02-25 00:49:20+00:00,0,This doesn't surprise me since the entire hist...,qpClEvsjW0s,True,This doesn't surprise me since the entire hist...,-0.0121,Negative,RoBERTa
1,@andrerosekriel1127,2026-02-22 11:15:55+00:00,0,Okay then why work for these factories.....or ...,qpClEvsjW0s,True,Okay then why work for these factories.....or ...,-0.2732,Negative,VADER
2,@sbradshaw1886,2026-03-17 02:12:15+00:00,0,Please do not be this ignorant. The US and oth...,qpClEvsjW0s,True,Please do not be this ignorant. The US and oth...,-0.0194,Negative,RoBERTa


## 2. How Language Detection Works

Before running on all rows, let's see `langdetect` in action on real examples.


In [3]:
examples = [
    "Shein is destroying the environment and exploiting workers",
    "Shein è un disastro per l'ambiente e sfrutta i lavoratori",   # Italian
    "Shein está destruyendo el medio ambiente",                     # Spanish
    "Shein détruit l'environnement",                               # French
    "ok",                                                          # too short to detect reliably
]

print("Language detection examples:\n")
for text in examples:
    try:
        lang = detect(text)
    except LangDetectException:
        lang = "unknown (too short)"
    print(f"  [{lang}] {text}")

Language detection examples:

  [en] Shein is destroying the environment and exploiting workers
  [it] Shein è un disastro per l'ambiente e sfrutta i lavoratori
  [es] Shein está destruyendo el medio ambiente
  [fr] Shein détruit l'environnement
  [sk] ok


## 3. Run Language Detection on All 39,422 Comments

This takes about 3–4 minutes. We add a `language` column to every row.

We label short/ambiguous comments as `'unknown'` — these will also be removed
since we can't confirm they're English.


In [4]:
def detect_lang(text):
    """Detect language of a comment. Returns language code or 'unknown'."""
    try:
        return detect(str(text))
    except LangDetectException:
        # langdetect throws an error when text is too short or ambiguous
        return 'unknown'

# Apply to every row (~3-4 minutes)
print("Running language detection... (this takes a few minutes)")
df['language'] = df['text_clean'].apply(detect_lang)
print("Done!")

# Show the full language breakdown
print(f"\nLanguage distribution:")
lang_counts = df['language'].value_counts()
total = len(df)
for lang, count in lang_counts.head(15).items():
    print(f"  {lang:10s}: {count:,} ({count/total*100:.1f}%)")

Running language detection... (this takes a few minutes)
Done!

Language distribution:
  en        : 30,609 (77.6%)
  it        : 2,399 (6.1%)
  es        : 1,479 (3.8%)
  pt        : 809 (2.1%)
  de        : 511 (1.3%)
  fr        : 460 (1.2%)
  unknown   : 286 (0.7%)
  ru        : 260 (0.7%)
  af        : 207 (0.5%)
  tl        : 188 (0.5%)
  id        : 178 (0.5%)
  so        : 173 (0.4%)
  zh-cn     : 165 (0.4%)
  tr        : 136 (0.3%)
  nl        : 132 (0.3%)


## 4. Filter — Keep Only English Comments

We keep rows where `language == 'en'` and drop everything else.


In [5]:
before = len(df)

# Keep only English
df_english = df[df['language'] == 'en'].copy()
df_english = df_english.reset_index(drop=True)

after = len(df_english)
dropped = before - after

print(f"Before filtering : {before:,} comments")
print(f"After filtering  : {after:,} comments")
print(f"Removed          : {dropped:,} non-English comments ({dropped/before*100:.1f}%)")
print(f"\nLanguages removed:")
removed = df[df['language'] != 'en']['language'].value_counts()
for lang, count in removed.head(10).items():
    print(f"  {lang}: {count:,}")

Before filtering : 39,422 comments
After filtering  : 30,609 comments
Removed          : 8,813 non-English comments (22.4%)

Languages removed:
  it: 2,399
  es: 1,479
  pt: 809
  de: 511
  fr: 460
  unknown: 286
  ru: 260
  af: 207
  tl: 188
  id: 178


## 5. Verify Sentiment Still Looks Correct

After dropping non-English comments, let's confirm the sentiment distribution
is still reasonable — it shouldn't change dramatically.


In [6]:
print("Sentiment distribution after language filtering:")
print(df_english['sentiment_label'].value_counts())
print()
print("As percentages:")
print(df_english['sentiment_label'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

Sentiment distribution after language filtering:
sentiment_label
Positive    14554
Negative    11958
Neutral      4097
Name: count, dtype: int64

As percentages:
sentiment_label
Positive    47.5%
Negative    39.1%
Neutral     13.4%
Name: proportion, dtype: object


## 6. Save the Final Filtered Dataset

In [7]:
# Save — this is now the MAIN dataset for all future steps
output_path = 'youtube_comments_english.csv'
df_english.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Total rows: {len(df_english):,}")
print(f"Columns: {df_english.columns.tolist()}")
print("\nPreview:")
df_english[['author', 'text_clean', 'sentiment_label', 'language']].head(5)

Saved to: youtube_comments_english.csv
Total rows: 30,609
Columns: ['author', 'updated_at', 'like_count', 'text', 'video_id', 'public', 'text_clean', 'sentiment_score', 'sentiment_label', 'sentiment_source', 'language']

Preview:


,author,text_clean,sentiment_label,language
0,@joaquinrodriguezvillegas366,This doesn't surprise me since the entire hist...,Negative,en
1,@andrerosekriel1127,Okay then why work for these factories.....or ...,Negative,en
2,@sbradshaw1886,Please do not be this ignorant. The US and oth...,Negative,en
3,@andrerosekriel1127,"@sbradshaw1886 Still a choice ...Ignorence, re...",Neutral,en
4,@RlxExplorer,They fuel buying addiction. Same with Temu. No...,Negative,en


## Summary

**What we did in this step:**
- Detected the language of all 39,422 comments using `langdetect`
- Found that ~23% of comments were non-English (Italian, Spanish, Portuguese, French, German, etc.)
- Removed all non-English and unknown-language comments
- Saved the result as `youtube_comments_english.csv`

**Final dataset going forward: ~30,609 English comments**

This file is the input for all remaining steps:
- Step 3: Keyword Extraction & NER
- Step 4: Topic Modeling
- Step 5: RAG System
- Step 6: Dashboard

**Note for report:** *"The analysis was scoped to English-language comments (n = 30,609) to ensure compatibility with English NLP models (VADER, spaCy, BERTopic). Non-English comments accounted for approximately 23% of the dataset and were excluded."*
